<a href="https://colab.research.google.com/github/Svein-Tore/colab/blob/main/FOPDT-binder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatSlider, Button, VBox, HBox, Output, FileUpload
from IPython.display import display, Markdown
import io

# === Oppsett for filopplasting ===
uploader = FileUpload(accept='.csv', multiple=False, description="Last opp CSV")
main_output = Output()

def start_analysen(change):
    with main_output:
        main_output.clear_output()

        # Hent filen som er lastet opp
        if not uploader.value:
            print("Vennligst last opp en fil først.")
            return

        # Hent data fra opplastet fil (fungerer i både MyBinder og Colab)
        file_item = uploader.value[0]
        content = file_item['content']
        df = pd.read_csv(io.BytesIO(content), sep=None, engine='python', decimal=',')

        tid_data = df.iloc[:,0].values
        niva_data = df.iloc[:,1].values

        # --- Automatisk estimering ---
        y0_est = niva_data[0]
        A_est = niva_data[-1] - y0_est
        thresh10 = y0_est + 0.1 * A_est
        thresh85 = y0_est + 0.85 * A_est

        L_est10 = tid_data[np.where(niva_data > thresh10)[0][0]]
        L_est85 = tid_data[np.where(niva_data > thresh85)[0][0]]
        L_est = abs(L_est10 - 0.05 * (L_est85 - L_est10))
        thresh63 = y0_est + 0.63 * A_est
        T_est = tid_data[np.where(niva_data > thresh63)[0][0]] - L_est

        display(Markdown(f"**Autoestimat:** $\Delta y$={A_est:.2f}, T={T_est:.2f}, L={L_est:.2f}, y0={y0_est:.2f}"))

        # --- Plott-funksjon ---
        def plot_fopdt(A, T, L, y0, save_png=False):
            y_model = np.where(tid_data < L, y0, y0 + A * (1 - np.exp(-(tid_data - L) / T)))
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(tid_data, niva_data, "b.", markersize=3, label="Måledata")
            ax.plot(tid_data, y_model, "r-", label=f"Modell: $\Delta y=${A:.2f}, T={T:.2f}")

            # Hjelpelinjer og pynt
            ax.set_xlabel("Tid [s]"); ax.set_ylabel("Nivå / respons")
            ax.grid(which='both'); ax.legend()

            if save_png:
                plt.savefig("FOPDT_plot.png", dpi=300)
                print("Plott lagret som FOPDT_plot.png i filoversikten.")

            plt.show()

        # --- Widgets for interaktivitet ---
        A_slider = FloatSlider(value=A_est, min=0, max=2*A_est, description="Δy")
        T_slider = FloatSlider(value=T_est, min=0.1, max=2*T_est, description="T")
        L_slider = FloatSlider(value=L_est, min=0.01, max=8*L_est, description="L")
        y0_slider = FloatSlider(value=y0_est, min=0, max=2*y0_est, description="y0")
        save_btn = Button(description="Lagre PNG", button_style='success')

        plot_out = Output()

        def update_plot(change):
            with plot_out:
                plot_out.clear_output(wait=True)
                plot_fopdt(A_slider.value, T_slider.value, L_slider.value, y0_slider.value)

        for s in [A_slider, T_slider, L_slider, y0_slider]:
            s.observe(update_plot, "value")

        save_btn.on_click(lambda b: plot_fopdt(A_slider.value, T_slider.value, L_slider.value, y0_slider.value, True))

        display(VBox([plot_out, A_slider, T_slider, L_slider, y0_slider, save_btn]))
        update_plot(None)

# Startknapp for å trigge analysen etter opplasting
uploader.observe(start_analysen, names='value')

display(Markdown("### 1. Last opp måledata (.csv)"), uploader, main_output)